# ECC Volatility Prediction

It assumes the earnings-call
QA embeddings (with `ticker`/`date` fields already attached by the data-pipeline
notebook) and the KeFVP price/volatility label CSVs are available locally, and it
trains and compares several volatility-prediction models built on top of them.



## Setup & configuration

In [1]:
import os
import pickle
import random
from collections import defaultdict
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_DIR = Path("./data/volatility_embeddings")
RESULTS_DIR = Path("./results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [1, 3, 5, 6, 9, 11, 13, 14, 15, 17, 18, 19,
         21, 23, 24, 26, 28, 30, 31, 32, 33, 34, 35,
         39, 40, 42, 43, 44, 45, 46, 48, 50, 53, 59,
         60, 61, 63, 64, 65, 66, 67, 69, 71, 73, 76,
         80, 81, 83, 87, 88, 89, 93, 94, 96, 98]

BATCH_SIZE = 32
QA_EMBED_DIM = 3584
HORIZONS = [3, 7, 15, 30]
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SCHED_PATIENCE = 10
SCHED_FACTOR = 0.5
SCHED_MIN_LR = 1e-6
EARLY_STOP_PATIENCE = 100
NUM_EPOCHS = 100

past_cols = [f"past_{i}" for i in range(2, 30)]
target_cols = [f"future_{h}" for h in HORIZONS]

Using device: cuda


## Load price / volatility label CSVs

In [2]:
!wget -q -O maec15_train_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/15/maec15_train_avg_val.csv
!wget -q -O maec15_dev_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/15/maec15_dev_avg_val.csv
!wget -q -O maec15_test_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/15/maec15_test_avg_val.csv

!wget -q -O maec16_train_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/16/maec16_train_avg_val.csv
!wget -q -O maec16_dev_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/16/maec16_dev_avg_val.csv
!wget -q -O maec16_test_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/16/maec16_test_avg_val.csv

!wget -q -O ec_test_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/test_split_Avg_Series_WITH_LOG.csv

In [3]:
maec15_train_avg_val_df = pd.read_csv("maec15_train_avg_val.csv")
maec15_dev_avg_val_df = pd.read_csv("maec15_dev_avg_val.csv")
maec15_test_avg_val_df = pd.read_csv("maec15_test_avg_val.csv")

maec16_train_avg_val_df = pd.read_csv("maec16_train_avg_val.csv")
maec16_dev_avg_val_df = pd.read_csv("maec16_dev_avg_val.csv")
maec16_test_avg_val_df = pd.read_csv("maec16_test_avg_val.csv")

ec_test_avg_val_df = pd.read_csv("ec_test_avg_val.csv")

maec_combined_train_df = pd.concat(
    [maec15_train_avg_val_df, maec16_train_avg_val_df], ignore_index=True
)
maec_combined_dev_df = pd.concat(
    [maec15_dev_avg_val_df, maec16_dev_avg_val_df], ignore_index=True
)

PRICE_DFS = {
    "maec15": {
        "train": maec15_train_avg_val_df,
        "dev": maec15_dev_avg_val_df,
        "test": maec15_test_avg_val_df,
    },
    "maec16": {
        "train": maec16_train_avg_val_df,
        "dev": maec16_dev_avg_val_df,
        "test": maec16_test_avg_val_df,
    },
    "ec": {
        "train": maec_combined_train_df,
        "dev": maec_combined_dev_df,
        "test": ec_test_avg_val_df,
    },
}


## Load embeddings

One pickle per `(dataset, split, modality)`. `MODALITY_SUFFIX` maps each of the
6 modalities to its filename suffix; EC only has a `test` split.


In [6]:
MODALITY_SUFFIX = {
    "text_audio_prompt1": "embeddings",
    "text_only": "text_embeddings",
    "text_audio_prompt2": "qa_text_audio_prompt2_embeddings",
    "audio_only": "audio_embeddings",
    "chunk_text_audio": "chunk_text_audio_embeddings",
}

MODALITIES = list(MODALITY_SUFFIX)


def load_embeddings(dataset: str, split: str, modality: str):
    """Load a single embeddings pickle file.

    dataset: "maec15" | "maec16" | "ec"
    split:   "train" | "dev" | "test"  (EC only has "test")
    modality: one of MODALITY_SUFFIX
    """
    suffix = MODALITY_SUFFIX[modality]
    fname = f"kefvp_{split}_{dataset}_{suffix}.pkl"
    path = DATA_DIR / fname
    with open(path, "rb") as f:
        return pickle.load(f)


# embeddings[(dataset, split, modality)] -> list[dict] with "ticker"/"date"/"embedding"
EMBEDDINGS = {}

for _dataset in ("maec15", "maec16"):
    for _split in ("train", "dev", "test"):
        for _modality in MODALITIES:
            EMBEDDINGS[(_dataset, _split, _modality)] = load_embeddings(
                _dataset, _split, _modality
            )

for _modality in MODALITIES:
    EMBEDDINGS[("ec", "test", _modality)] = load_embeddings("ec", "test", _modality)

print(f"Loaded {len(EMBEDDINGS)} embedding files.")

Loaded 35 embedding files.


In [7]:
def build_lookup(embedding_lists):
    """Index a list of embedding-lists by (ticker, date) -> list[np.ndarray]."""
    lookup = defaultdict(list)
    for dataset in embedding_lists:
        for item in dataset:
            key = (item["ticker"], str(item["date"]))
            lookup[key].append(item["embedding"].astype(np.float32))
    return lookup


def get_qa_lists(df, lookup, is_ec: bool = False):
    """For each row of a price/label dataframe, return the list of QA embeddings
    (possibly empty) whose (ticker, date) matches that row."""
    result = []

    for _, row in df.iterrows():
        if is_ec and row.name in {94, 42}: # remove QAs of these two ( ECL, 2017, 10, 31 , NTAP, 2017, 11, 15)
            print('removed', row.name)
            continue

        ticker = row["ticker"]

        if is_ec:
            date = f"{int(row['year']):04d}{int(row['month']):02d}{int(row['day']):02d}"
        else:
            date = str(row["time"]).replace("-", "")

        key = (ticker, date)
        qas = lookup.get(key, [])
        if len(qas) == 0:
            continue
        result.append(qas)

    return result




def get_data_and_qa_for(dataset_key: str, split: str, modality: str):
    """
    Return a price/label dataframe and its corresponding QA embeddings,
    aligned row-by-row.
    """

    df = PRICE_DFS[dataset_key][split].copy()

    if dataset_key == "ec":
        if split == "test":
            embedding_lists = [
                EMBEDDINGS[("ec", "test", modality)]
            ]
            is_ec = True
        else:
            embedding_lists = [
                EMBEDDINGS[("maec15", split, modality)],
                EMBEDDINGS[("maec16", split, modality)],
            ]
            is_ec = False
    else:
        embedding_lists = [
            EMBEDDINGS[(dataset_key, split, modality)]
        ]
        is_ec = False

    lookup = build_lookup(embedding_lists)

    rows = []
    qa_lists = []

    for _, row in df.iterrows():

        ticker = row["ticker"]

        if is_ec:
            date = (
                f"{int(row['year']):04d}"
                f"{int(row['month']):02d}"
                f"{int(row['day']):02d}"
            )
        else:
            date = str(row["time"]).replace("-", "")

        qas = lookup.get((ticker, date), [])

        if not qas:
            continue

        rows.append(row)
        qa_lists.append(qas)

    aligned_df = pd.DataFrame(rows).reset_index(drop=True)

    assert len(aligned_df) == len(qa_lists)

    return aligned_df, qa_lists

## Shared dataset & model definitions

In [8]:
class VolDataset(Dataset):
    """Volatility-history-only dataset, used for the vol-only baseline."""

    def __init__(self, df):
        self.X = df[past_cols].values.astype(np.float32)
        self.y = df[target_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class ECCDataset(Dataset):
    """Volatility history + variable-length list of QA embeddings per example.

    Used for every fusion/QA-only experiment. The "whole chunk" (no-QA) modality
    reuses this same class -- each example's "QA list" there is just a length-1
    list containing the single chunk embedding.
    """

    def __init__(self, df, qa_lists):
        self.X = df[past_cols].values.astype(np.float32)
        self.y = df[target_cols].values.astype(np.float32)
        self.qa_lists = qa_lists

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.qa_lists[idx], self.y[idx]


def collate_fn(batch):
    """Pad the variable-length QA lists in a batch to a common length and build a
    boolean mask marking real (non-padding) positions."""
    vols, qas, ys = zip(*batch)

    vols = torch.tensor(np.stack(vols), dtype=torch.float32)
    ys = torch.tensor(np.stack(ys), dtype=torch.float32)

    max_len = max(max(len(x), 1) for x in qas)

    qa_tensor = torch.zeros(len(qas), max_len, QA_EMBED_DIM, dtype=torch.float32)
    mask = torch.zeros(len(qas), max_len, dtype=torch.bool)

    for i, qa_list in enumerate(qas):
        if len(qa_list) == 0:
            continue
        arr = np.stack(qa_list)
        qa_tensor[i, : len(arr)] = torch.tensor(arr)
        mask[i, : len(arr)] = True

    return vols, qa_tensor, mask, ys

In [9]:
class AttentionPooling(nn.Module):
    """Masked attention pooling over a padded sequence of QA embeddings."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)

        weights = torch.softmax(scores, dim=1)
        weights = weights * mask.float()
        weights = weights / (weights.sum(dim=1, keepdim=True) + 1e-8)

        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled

In [10]:
class VolOnlyModel(nn.Module):
    """Volatility-history-only baseline: 2-layer LSTM over `past_cols`, MLP head.

    This is the frozen baseline whose `vol_lstm` + `head` get reused (frozen) by
    `VolECCResidualGatedModel`.
    """

    def __init__(self):
        super().__init__()

        self.vol_lstm = nn.LSTM(
            input_size=1, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2
        )

        self.head = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
        )

    def forward(self, vol):
        vol = vol.unsqueeze(-1)  # [B, 28] -> [B, 28, 1]
        _, (h, _) = self.vol_lstm(vol)
        vol_repr = h[-1]  # [B, 64]
        return self.head(vol_repr)

In [11]:
class QAOnlyModel(nn.Module):
    """QA-embeddings-only model: no volatility branch at all."""

    def __init__(self):
        super().__init__()

        self.qa_proj = nn.Sequential(
            nn.Linear(QA_EMBED_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
        )

        self.qa_lstm = nn.LSTM(
            input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True
        )

        self.attention_pool = AttentionPooling(hidden_dim=256)

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
        )

    def forward(self, qa, mask):
        qa = self.qa_proj(qa)
        qa_out, _ = self.qa_lstm(qa)
        qa_repr = self.attention_pool(qa_out, mask)
        return self.head(qa_repr)

In [12]:
class VolECCConcatModel(nn.Module):
    """Concat fusion: vol_lstm (trained from scratch) || (qa_proj -> BiLSTM ->
    AttentionPooling), concatenated and passed through an MLP head."""

    def __init__(self):
        super().__init__()

        self.vol_lstm = nn.LSTM(
            input_size=1, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2
        )

        self.qa_proj = nn.Sequential(
            nn.Linear(QA_EMBED_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
        )

        self.qa_lstm = nn.LSTM(
            input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True
        )

        self.attention_pool = AttentionPooling(hidden_dim=256)

        self.head = nn.Sequential(
            nn.Linear(64 + 256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
        )

    def forward(self, vol, qa, mask):
        vol = vol.unsqueeze(-1)
        _, (h, _) = self.vol_lstm(vol)
        vol_repr = h[-1]

        qa = self.qa_proj(qa)
        qa_out, _ = self.qa_lstm(qa)
        qa_repr = self.attention_pool(qa_out, mask)

        fused = torch.cat([vol_repr, qa_repr], dim=-1)
        out = self.head(fused)
        return out

In [ ]:
# class VolECCGatedNoBiLSTMModel(nn.Module):
#     """Gated fusion, no BiLSTM on the QA branch: attention pooling is applied
#     directly to `qa_proj`'s output, and a learned gate scales the pooled QA
#     representation before concatenation with the volatility representation."""

#     def __init__(self):
#         super().__init__()

#         self.vol_lstm = nn.LSTM(
#             input_size=1, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2
#         )

#         self.qa_proj = nn.Sequential(
#             nn.Linear(QA_EMBED_DIM, 512),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(512, 256),
#         )

#         self.attention_pool = AttentionPooling(hidden_dim=256)

#         self.qa_gate = nn.Sequential(
#             nn.Linear(64 + 256, 128),
#             nn.ReLU(),
#             nn.Linear(128, 256),
#             nn.Sigmoid(),
#         )

#         self.head = nn.Sequential(
#             nn.Linear(64 + 256, 128),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(128, 64),
#             nn.ReLU(),
#             nn.Linear(64, 4),
#         )

#     def forward(self, vol, qa, mask):
#         vol = vol.unsqueeze(-1)
#         _, (h, _) = self.vol_lstm(vol)
#         vol_repr = h[-1]

#         qa = self.qa_proj(qa)
#         qa_repr = self.attention_pool(qa, mask)  # attention directly on qa_proj output

#         gate = self.qa_gate(torch.cat([vol_repr, qa_repr], dim=-1))
#         qa_repr = gate * qa_repr

#         fused = torch.cat([vol_repr, qa_repr], dim=-1)
#         out = self.head(fused)
#         return out

In [13]:
class VolECCResidualGatedModel(nn.Module):
    """Residual/frozen fusion: a frozen, pretrained `VolOnlyModel` (`vol_lstm` +
    `head`, both `requires_grad=False`) predicts a base volatility forecast; a QA
    branch (proj -> BiLSTM -> AttentionPooling -> qa_reduce) predicts a delta on
    top of it. `delta_head`'s final layer is zero-initialized so the model starts
    out numerically identical to the frozen vol-only baseline and has to learn to
    deviate from it.
    """

    def __init__(self, vol_model: "VolOnlyModel"):
        super().__init__()

        # Reuse the pretrained volatility branch, frozen.
        self.vol_lstm = vol_model.vol_lstm
        self.vol_head = vol_model.head
        for p in self.vol_lstm.parameters():
            p.requires_grad = False
        for p in self.vol_head.parameters():
            p.requires_grad = False

        self.qa_proj = nn.Sequential(
            nn.Linear(QA_EMBED_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
        )

        self.qa_lstm = nn.LSTM(
            input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True
        )

        self.attention_pool = AttentionPooling(hidden_dim=256)

        self.qa_reduce = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
        )

        self.delta_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 4),
        )
        nn.init.zeros_(self.delta_head[-1].weight)
        nn.init.zeros_(self.delta_head[-1].bias)

    def forward(self, vol, qa, mask):
        vol = vol.unsqueeze(-1)
        _, (h, _) = self.vol_lstm(vol)
        vol_repr = h[-1]

        qa = self.qa_proj(qa)
        qa_out, _ = self.qa_lstm(qa)
        qa_repr = self.attention_pool(qa_out, mask)
        qa_repr = self.qa_reduce(qa_repr)

        vol_pred = self.vol_head(vol_repr)
        delta_input = torch.cat([vol_repr, qa_repr], dim=-1)
        delta = self.delta_head(delta_input)

        out = vol_pred + delta
        return out, vol_pred, delta


FUSION_MODEL_CLASSES = {
    "concat": VolECCConcatModel,
    # "gated_no_bilstm": VolECCGatedNoBiLSTMModel,
    "residual_gated": VolECCResidualGatedModel,
    "qa_only": QAOnlyModel,
}

## Vol-only baseline pretraining

Trains `VolOnlyModel` per seed for each dataset key that a fusion experiment will
later need frozen weights from.

In [17]:
def train_vol_only_baseline(dataset_key: str, seeds=SEEDS):
    """Train VolOnlyModel once per seed for one dataset key, returning
    {seed: state_dict} and a per-seed results dataframe."""

    train_df = PRICE_DFS[dataset_key]["train"]
    dev_df = PRICE_DFS[dataset_key]["dev"]
    test_df = PRICE_DFS[dataset_key]["test"]

    best_states = {}
    all_results = []

    for seed in seeds:
        print(f"\n[vol_only:{dataset_key}] seed {seed}")

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_loader = DataLoader(VolDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
        dev_loader = DataLoader(VolDataset(dev_df), batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(VolDataset(test_df), batch_size=BATCH_SIZE, shuffle=False)

        model = VolOnlyModel().to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )

        best_val = float("inf")
        best_state = None
        counter = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            for vol, y in train_loader:
                vol, y = vol.to(device), y.to(device)
                optimizer.zero_grad()
                pred = model(vol)
                loss = criterion(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for vol, y in dev_loader:
                    vol, y = vol.to(device), y.to(device)
                    val_loss += criterion(model(vol), y).item()
            val_loss /= len(dev_loader)

            if val_loss < best_val:
                best_val = val_loss
                best_state = deepcopy(model.state_dict())
                counter = 0
            else:
                counter += 1
                if counter >= EARLY_STOP_PATIENCE:
                    break

        best_states[seed] = best_state
        model.load_state_dict(best_state)

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for vol, y in test_loader:
                vol = vol.to(device)
                preds.append(model(vol).cpu().numpy())
                targets.append(y.numpy())
        preds = np.concatenate(preds)
        targets = np.concatenate(targets)

        mses = [
            mean_squared_error(targets[:, i], preds[:, i]) for i in range(len(HORIZONS))
        ]
        avg_mse = float(np.mean(mses))
        print(f"  avg_mse={avg_mse:.6f}")

        result = {"seed": seed, "avg_mse": avg_mse}
        for i, hrz in enumerate(HORIZONS):
            result[f"mse_{hrz}"] = mses[i]
        all_results.append(result)

        torch.save(best_state, RESULTS_DIR / f"vol_only_state_{dataset_key}_{seed}.pt")

    results_df = pd.DataFrame(all_results)
    results_df.to_csv(RESULTS_DIR / f"results_{dataset_key}_vol_only.csv", index=False)
    return best_states, results_df


VOL_ONLY_DATASET_KEYS = ["maec15", "maec16", "ec"]

best_models = {}
vol_only_results = {}

for _dataset_key in VOL_ONLY_DATASET_KEYS:
    _states, _results = train_vol_only_baseline(_dataset_key)
    best_models[_dataset_key] = _states
    vol_only_results[_dataset_key] = _results

print({k: v["avg_mse"].mean() for k, v in vol_only_results.items()})


[vol_only:maec15] seed 1
  avg_mse=0.188646

[vol_only:maec15] seed 3
  avg_mse=0.187581

[vol_only:maec15] seed 5
  avg_mse=0.189217

[vol_only:maec15] seed 6
  avg_mse=0.208927

[vol_only:maec15] seed 9
  avg_mse=0.188149

[vol_only:maec15] seed 11
  avg_mse=0.206596

[vol_only:maec15] seed 13
  avg_mse=0.192870

[vol_only:maec15] seed 14
  avg_mse=0.190261

[vol_only:maec15] seed 15
  avg_mse=0.200150

[vol_only:maec15] seed 17
  avg_mse=0.193388

[vol_only:maec15] seed 18
  avg_mse=0.195649

[vol_only:maec15] seed 19
  avg_mse=0.219245

[vol_only:maec15] seed 21
  avg_mse=0.203540

[vol_only:maec15] seed 23
  avg_mse=0.191448

[vol_only:maec15] seed 24
  avg_mse=0.205304

[vol_only:maec15] seed 26
  avg_mse=0.209660

[vol_only:maec15] seed 28
  avg_mse=0.194551

[vol_only:maec15] seed 30
  avg_mse=0.189450

[vol_only:maec15] seed 31
  avg_mse=0.193635

[vol_only:maec15] seed 32
  avg_mse=0.215996

[vol_only:maec15] seed 33
  avg_mse=0.201117

[vol_only:maec15] seed 34
  avg_mse=0.

## Generic experiment runner

In [16]:
def run_experiment(
    name: str,
    dataset_key: str,
    modality: str,
    fusion_type: str,
    train_qas,
    dev_qas,
    test_qas,
    train_df,
    dev_df,
    test_df,
    needs_vol_pretrain: bool = True,
    seeds=SEEDS,
):
    """Train+evaluate one (dataset, modality, fusion_type) combination across all
    seeds, returning a per-seed results dataframe and also writing it to
    RESULTS_DIR.

    fusion_type: "concat"   | "residual_gated" | "qa_only"
    needs_vol_pretrain: True for "residual_gated" (loads the frozen VolOnlyModel
        weights trained in the previous section); ignored otherwise.
    """
    model_cls = FUSION_MODEL_CLASSES[fusion_type]
    all_results = []

    for seed in seeds:
        print(f"\n[{name}] seed {seed}")

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_loader = DataLoader(
            ECCDataset(train_df, train_qas), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
        )
        dev_loader = DataLoader(
            ECCDataset(dev_df, dev_qas), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
        )
        test_loader = DataLoader(
            ECCDataset(test_df, test_qas), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
        )

        if fusion_type == "residual_gated" and needs_vol_pretrain:
            vol_model = VolOnlyModel().to(device)
            vol_model.load_state_dict(best_models[dataset_key][seed])
            model = model_cls(vol_model).to(device)
        else:
            model = model_cls().to(device)

        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=SCHED_FACTOR,
            patience=SCHED_PATIENCE,
            threshold=1e-4,
            min_lr=SCHED_MIN_LR,
        )

        best_val = float("inf")
        best_state = None
        counter = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            for vol, qa, mask, y in train_loader:
                vol, qa, mask, y = vol.to(device), qa.to(device), mask.to(device), y.to(device)
                optimizer.zero_grad()

                if fusion_type == "residual_gated":
                    pred, _, _ = model(vol, qa, mask)
                elif fusion_type == "qa_only":
                    pred = model(qa, mask)
                else:
                    pred = model(vol, qa, mask)

                loss = criterion(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for vol, qa, mask, y in dev_loader:
                    vol, qa, mask, y = vol.to(device), qa.to(device), mask.to(device), y.to(device)
                    if fusion_type == "residual_gated":
                        pred, _, _ = model(vol, qa, mask)
                    elif fusion_type == "qa_only":
                        pred = model(qa, mask)
                    else:
                        pred = model(vol, qa, mask)
                    val_loss += criterion(pred, y).item()
            val_loss /= len(dev_loader)
            scheduler.step(val_loss)

            if val_loss < best_val:
                best_val = val_loss
                best_state = deepcopy(model.state_dict())
                counter = 0
            else:
                counter += 1
                if counter >= EARLY_STOP_PATIENCE:
                    break

        model.load_state_dict(best_state)
        model.eval()

        preds, targets = [], []
        with torch.no_grad():
            for vol, qa, mask, y in test_loader:
                vol, qa, mask = vol.to(device), qa.to(device), mask.to(device)
                if fusion_type == "residual_gated":
                    pred, _, _ = model(vol, qa, mask)
                elif fusion_type == "qa_only":
                    pred = model(qa, mask)
                else:
                    pred = model(vol, qa, mask)
                preds.append(pred.cpu().numpy())
                targets.append(y.numpy())

        preds = np.concatenate(preds)
        targets = np.concatenate(targets)

        mses = [
            mean_squared_error(targets[:, i], preds[:, i]) for i in range(len(HORIZONS))
        ]
        avg_mse = float(np.mean(mses))
        print(f"  avg_mse={avg_mse:.6f}")

        result = {
            "seed": seed,
            "dataset": dataset_key,
            "modality": modality,
            "fusion_type": fusion_type,
            "avg_mse": avg_mse,
        }
        for i, hrz in enumerate(HORIZONS):
            result[f"mse_{hrz}"] = mses[i]
        all_results.append(result)

    results_df = pd.DataFrame(all_results)


    if fusion_type == "residual_gated":
        csv_name = f"results_{dataset_key}_{modality}.csv"
    else:
        csv_name = f"results_{dataset_key}_{modality}_{fusion_type}.csv"

    results_df.to_csv(RESULTS_DIR / csv_name, index=False)
    return results_df

## Experiment configuration table

Covers every `{modality} x {dataset} x {fusion_type}` combination, plus one extra
`qa_only` row per dataset for the `text_audio_prompt2` modality.



In [17]:

DATASETS = ["maec15", "maec16", "ec"]
FUSION_TYPES = ["residual_gated"]

PROMPT2 = "text_audio_prompt2"


EXPERIMENT_CONFIG = []

for modality in MODALITIES:
    for dataset in DATASETS:
        for fusion_type in FUSION_TYPES:
            EXPERIMENT_CONFIG.append({
                "dataset": dataset,
                "modality": modality,
                "fusion_type": fusion_type,
            })

    if modality == PROMPT2:
        for dataset in DATASETS:
            EXPERIMENT_CONFIG.append({
                "dataset": dataset,
                "modality": modality,
                "fusion_type": "qa_only",
            })
            EXPERIMENT_CONFIG.append({
                "dataset": dataset,
                "modality": modality,
                "fusion_type": "concat",
            })


experiment_config_df = pd.DataFrame(EXPERIMENT_CONFIG)
print(f"{len(experiment_config_df)} experiment rows")
experiment_config_df

9 experiment rows


,dataset,modality,fusion_type
0,maec15,text_audio_prompt2,residual_gated
1,maec16,text_audio_prompt2,residual_gated
2,ec,text_audio_prompt2,residual_gated
3,maec15,text_audio_prompt2,qa_only
4,maec15,text_audio_prompt2,concat
5,maec16,text_audio_prompt2,qa_only
6,maec16,text_audio_prompt2,concat
7,ec,text_audio_prompt2,qa_only
8,ec,text_audio_prompt2,concat


## Run all experiments

Iterates the config table above, loads the right QA lists for each
`(dataset, modality)` pair once, and calls `run_experiment` for each row. Results
from every row are concatenated into one `all_results_df`; each row also writes
its own CSV via `run_experiment`.

In [18]:
all_results = []
_qa_cache = {}

for _, cfg in experiment_config_df.iterrows():
    dataset_key = cfg["dataset"]
    modality = cfg["modality"]
    fusion_type = cfg["fusion_type"]
    name = f"{dataset_key}_{modality}_{fusion_type}"

    cache_key = (dataset_key, modality)
    if cache_key not in _qa_cache:
        _qa_cache[cache_key] = {
            "train": get_data_and_qa_for(dataset_key, "train", modality),
            "dev": get_data_and_qa_for(dataset_key, "dev", modality),
            "test": get_data_and_qa_for(dataset_key, "test", modality),
        }
    train_df, train_qas = _qa_cache[cache_key]["train"]
    dev_df, dev_qas = _qa_cache[cache_key]["dev"]
    test_df, test_qas = _qa_cache[cache_key]["test"]

    result_df = run_experiment(
        name=name,
        dataset_key=dataset_key,
        modality=modality,
        fusion_type=fusion_type,
        train_qas=train_qas,
        dev_qas=dev_qas,
        test_qas=test_qas,
        train_df=train_df,
        dev_df=dev_df,
        test_df=test_df,
        needs_vol_pretrain=(fusion_type == "residual_gated"),
    )
    all_results.append(result_df)

all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.to_csv(RESULTS_DIR / "all_results.csv", index=False)
all_results_df.head()


[maec15_text_audio_prompt2_residual_gated] seed 1
  avg_mse=0.186430

[maec15_text_audio_prompt2_residual_gated] seed 3
  avg_mse=0.183610

[maec15_text_audio_prompt2_residual_gated] seed 5
  avg_mse=0.186866

[maec15_text_audio_prompt2_residual_gated] seed 6
  avg_mse=0.208390

[maec15_text_audio_prompt2_residual_gated] seed 9
  avg_mse=0.187498

[maec15_text_audio_prompt2_residual_gated] seed 11
  avg_mse=0.203866

[maec15_text_audio_prompt2_residual_gated] seed 13
  avg_mse=0.191753

[maec15_text_audio_prompt2_residual_gated] seed 14
  avg_mse=0.187782

[maec15_text_audio_prompt2_residual_gated] seed 15
  avg_mse=0.199805

[maec15_text_audio_prompt2_residual_gated] seed 17
  avg_mse=0.191159

[maec15_text_audio_prompt2_residual_gated] seed 18
  avg_mse=0.191197

[maec15_text_audio_prompt2_residual_gated] seed 19
  avg_mse=0.217757

[maec15_text_audio_prompt2_residual_gated] seed 21
  avg_mse=0.203347

[maec15_text_audio_prompt2_residual_gated] seed 23
  avg_mse=0.187707

[maec15_te

KeyboardInterrupt: 

## Results aggregation & comparison

In [ ]:
mse_cols = [f"mse_{h}" for h in HORIZONS]

summary_df = (
    all_results_df
    .groupby(["dataset", "modality", "fusion_type"])[mse_cols + ["avg_mse"]]
    .agg(["mean", "std"])
)
summary_df

In [ ]:
vol_only_summary = {
    dataset_key: {
        "avg_mse_mean": df["avg_mse"].mean(),
        "avg_mse_std": df["avg_mse"].std(),
    }
    for dataset_key, df in vol_only_results.items()
}

comparison_rows = []
for dataset_key, stats in vol_only_summary.items():
    comparison_rows.append({
        "dataset": dataset_key,
        "modality": "vol_only",
        "fusion_type": "n/a",
        "avg_mse_mean": stats["avg_mse_mean"],
        "avg_mse_std": stats["avg_mse_std"],
    })

for (dataset_key, modality, fusion_type), group in all_results_df.groupby(
    ["dataset", "modality", "fusion_type"]
):
    comparison_rows.append({
        "dataset": dataset_key,
        "modality": modality,
        "fusion_type": fusion_type,
        "avg_mse_mean": group["avg_mse"].mean(),
        "avg_mse_std": group["avg_mse"].std(),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(["dataset", "avg_mse_mean"])
comparison_df

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6 * len(DATASETS), 5), sharey=True)

for ax, dataset_key in zip(axes, DATASETS):
    sub = comparison_df[comparison_df["dataset"] == dataset_key].copy()
    sub["label"] = sub["modality"] + sub["fusion_type"].apply(
        lambda f: "" if f == "n/a" else f"\n({f})"
    )
    sub = sub.sort_values("avg_mse_mean")

    ax.barh(sub["label"], sub["avg_mse_mean"], xerr=sub["avg_mse_std"].fillna(0.0))
    ax.set_title(dataset_key)
    ax.set_xlabel("Average MSE across horizons (lower is better)")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "modality_fusion_comparison.png", dpi=150)
plt.show()